In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
def data_parsing(CATAG,Year):
    path_dir = './source_data/cancer_mortality/'
    file_list = os.listdir(path_dir)

    cancer_mortal = pd.DataFrame()
    for i in file_list:
        if i.startswith('mortality_A') and (i.split('_')[3]).split('.')[0] == CATAG:

            file = './source_data/cancer_mortality/%s' % i
            x = pd.read_csv(file, sep = ',', header = 4)
            x = x.iloc[1:21,:]
            x = x.set_index('Unnamed: 0')
            x = x.replace({'-' : '0.0'}, regex=True)
            x = x.replace({',' : ''}, regex=True)
            x = x.astype(float)
            
            if Year == '2000-2019':
                empty = pd.DataFrame(index=['mean'], columns=x.columns)
                x = x.append(empty)
                for e in x.columns:
                    x.loc['mean'][e] = x[e][0:20].mean()
                x = x.iloc[20:21,:]
            elif Year == '2000':
                x = x.iloc[0:1,:]

            x['age'] = i.split('_')[2]
            x['order'] = [1 if x.iloc[0]['age'] == '14' 
                          else 2 if x.iloc[0]['age'] == '15-39' 
                          else 3 if x.iloc[0]['age'] == '40-64' 
                          else 4 if x.iloc[0]['age'] == '65-74' 
                          else 5]

            cancer_mortal = cancer_mortal.append(x)

    cancer_mortal = cancer_mortal.sort_values(by=['order'], ascending=[True])
    cancer_mortal = cancer_mortal.set_index('age')
    cancer_mortal = cancer_mortal.iloc[:,:-1]
    cancer_mortal = cancer_mortal.astype(float)
    
    return cancer_mortal


def mean(data):

    N = 140 if data.name == 'BOTH' else 152 if data.name == 'MALE' else 164
    
    df = pd.DataFrame()
    
    for i in range(0,N,4):
        y = data.iloc[:,i]
        df = pd.concat([df,y],axis=1)

    for i in df.columns:
        df[i] = (df[i] / df[i].sum()) * 100

    df = df.reset_index()
    df = pd.melt(df, id_vars=['index'])
    
    return df



def MERGE_sel(j):    
    
    A = j[j['variable'].isin(['All Cancer Sites Combined'])]

#    Y = j[j['variable'].isin(['Brain and Other Nervous System','Kidney and Renal Pelvis',
#                              'Melanoma of the Skin'])]
    Y = j[j['variable'].isin(['Brain and Other Nervous System','Kidney and Renal Pelvis','Colon and Rectum',
                              'Liver and Intrahepatic Bile Duct'])]

#    O = j[j['variable'].isin(['Lung and Bronchus','Cervix Uteri','Oral Cavity and Pharynx',
#                              'Oral Cavity and Pharynx','Larynx'])]
    O = j[j['variable'].isin(['Lung and Bronchus','Cervix Uteri','Oral Cavity and Pharynx','Melanoma of the Skin',
                              'Oral Cavity and Pharynx','Larynx'])]
    
        
    A['group'] = 'All combinded'
    Y['group'] = 'Young'
    O['group'] = 'Old'
    
    MERGE = pd.concat([A,Y,O],axis=0)
    
    A = A.rename(columns={'index':'Age','value':'Pertile','variable':'cancer types'})
    Y = Y.rename(columns={'index':'Age','value':'Pertile','variable':'cancer types'})
    O = O.rename(columns={'index':'Age','value':'Pertile','variable':'cancer types'})
    MERGE = MERGE.rename(columns={'index':'Age','value':'Pertile','variable':'cancer types'})
    
    return A,Y,O,MERGE

In [ ]:
def figures(MALE_Y,MALE_O,BOTH_M,MALE_M,FEMALE_M):
    
    fig = plt.figure(figsize=(7,7))
    gs = gridspec.GridSpec(nrows=2, ncols=2, height_ratios=[1,1],width_ratios=[1,19])

    ax1 = fig.add_subplot(gs[0,0])
    plt.xticks([])
    plt.yticks([])
    plt.text(1,0.23,'cancer_young',ha='right', size=15,rotation=90)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['left'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['bottom'].set_visible(False)

    ax3 = fig.add_subplot(gs[1,0])
    plt.xticks([])
    plt.yticks([])
    plt.text(1,0.3,'cancer_old',ha='right', size=15,rotation=90)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['left'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['bottom'].set_visible(False)


    ax2 = fig.add_subplot(gs[0,1])   
    sns.lineplot(data=MALE_Y,x='Age',y='Pertile',hue='cancer types',palette='Set1',ax=ax2)
    #ax2.get_legend().remove()
    plt.xlabel('')
    plt.ylabel('')
    plt.ylim(0,60)
    plt.xlim(0,4)

    ax4 = fig.add_subplot(gs[1,1])  
    sns.lineplot(data=MALE_O,x='Age',y='Pertile',hue='cancer types',palette='Set1',ax=ax4)
    #ax4.get_legend().remove()
    plt.ylabel('')
    plt.ylim(0,60)
    plt.xlim(0,4)

    plt.show()


    fig = plt.figure(figsize=(3.8,7))
    gs = gridspec.GridSpec(nrows=3, ncols=2, height_ratios=[1,1,1],width_ratios=[1,19])

    ax2 = fig.add_subplot(gs[0,1])    
    sns.lineplot(data=BOTH_M,x='Age',y='Pertile',hue='group',palette=['black','green','orange'],ax=ax2)
    #ax2.get_legend().remove()
    plt.xlabel('')
    plt.ylabel('')
    plt.ylim(0,70)
    plt.xlim(0,4)

    ax4 = fig.add_subplot(gs[1,1])    
    sns.lineplot(data=MALE_M,x='Age',y='Pertile',hue='group',palette=['black','green','orange'],ax=ax4)
    #ax4.get_legend().remove()
    plt.xlabel('')
    plt.ylabel('')
    plt.ylim(0,70)
    plt.xlim(0,4)

    ax6 = fig.add_subplot(gs[2,1])    
    sns.lineplot(data=FEMALE_M,x='Age',y='Pertile',hue='group',palette=['black','green','orange'],ax=ax6)
    #ax6.get_legend().remove()
    plt.ylabel('')
    plt.ylim(0,70)
    plt.xlim(0,4)

    ax1 = fig.add_subplot(gs[0,0])
    plt.xticks([])
    plt.yticks([])
    plt.text(0,0.25,'both sex',ha='right', size=15,rotation=90)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['left'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['bottom'].set_visible(False)


    ax3 = fig.add_subplot(gs[1,0])
    plt.xticks([])
    plt.yticks([])
    plt.text(0,0.38,'male',ha='right', size=15,rotation=90)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['left'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['bottom'].set_visible(False)

    ax5 = fig.add_subplot(gs[2,0])
    plt.xticks([])
    plt.yticks([])
    plt.text(0,0.33,'female',ha='right', size=15,rotation=90)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['left'].set_visible(False)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['bottom'].set_visible(False)

    plt.show()

In [ ]:
for i in ['2000-2019','2000']:
    
    print(i)
    
    both = data_parsing('both',i)
    male = data_parsing('male',i)
    female = data_parsing('female',i)

    both.name = 'BOTH'
    male.name = 'MALE'
    female.name = 'FEMALE'

    BOTH_P = mean(both)
    MALE_P = mean(male)
    FEMALE_P = mean(female)

    BOTH_A,BOTH_Y,BOTH_O,BOTH_M = MERGE_sel(BOTH_P)
    MALE_A,MALE_Y,MALE_O,MALE_M = MERGE_sel(MALE_P)
    FEMALE_A,FEMALE_Y,FEMALE_O,FEMALE_M = MERGE_sel(FEMALE_P)

    figures(MALE_Y,MALE_O,BOTH_M,MALE_M,FEMALE_M)